# Lab 5.1 &mdash; LangChain: rebuild the AskOps agent

**About 20 minutes** &middot; Day 2 &middot; Module 5 &mdash; LangChain &amp; LangGraph

On Day 1 you wrote the AskOps agent by hand, in `agent.py`. In this lab LangChain builds the same agent: a chain, three tools, and the whole loop in one call.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The model is real, so its words change from run to run. The shape of the result does not.

**The result:** the AskOps agent answers the two Lab 4.4 questions on the sandbox model, and you compare it with your Day 1 agent.

## Step 1 &mdash; Connect to the sandbox model

`get_llm()` returns a LangChain **chat model** for the sandbox model. It is one line, because the
sandbox already sets the address and the model name.

In [ ]:
from askops import get_llm, trace
import askops

llm = get_llm()
print("model:", llm.model_name)
print(llm.invoke("Greet an on-call engineer in five words.").content)

**You should see:** the model name, then a short greeting.

## Step 2 &mdash; A chain: `prompt | model | parser`

The pipe `|` passes the output of one part into the next. The prompt fills the `{text}` blank, the
model replies, and the parser returns plain text. Fixed steps with no tools make a **chain**.

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an SRE. Answer in one line."),
    ("user", "Summarise this incident for the team: {text}"),
])
chain = prompt | llm | StrOutputParser()

incidents = json.loads(askops.list_incidents())
print(chain.invoke({"text": json.dumps(incidents[0])}))

# batch: the same chain on several inputs at once
for line in chain.batch([{"text": json.dumps(i)} for i in incidents[1:]]):
    print("-", line)

**You should see:** a one-line summary of INC-9001, then two more lines for INC-9002 and INC-9003.
`batch` sent those two at the same time.

## Step 3 &mdash; Tools: a function plus a docstring

`@tool` turns a Python function into a tool. The **function name** becomes the tool name, the **type
hints** become the parameters, and the **docstring** becomes the description the model reads. In
Module 4 you wrote all of this by hand, in `TOOL_SPECS`.

`open_incident` is left out on purpose. It writes, so it needs a person's approval. Lab 5.3 adds it.

In [ ]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def search_runbooks(query: str, service: str | None = None) -> str:
    """Find runbooks that match a symptom, such as '502 after deploy'.
    Use for how-to-fix questions. Not for open incidents."""
    return askops.search_runbooks(query, service)

@tool
def get_runbook(runbook_id: str) -> str:
    """Return the steps of one runbook by id, such as 'RB-101'.
    Use after search_runbooks. Not for searching."""
    return askops.get_runbook(runbook_id)

@tool
def list_incidents() -> str:
    """List the open incidents, newest first."""
    return askops.list_incidents()

tools = [search_runbooks, get_runbook, list_incidents]
print(json.dumps(convert_to_openai_tool(get_runbook), indent=2))

**You should see:** the schema the model receives for `get_runbook`: its name, the docstring as the
description, and `runbook_id` as a required string. Open `labs/module-4-agent/agent.py` and compare
it with `TOOL_SPECS`. It is the same information, and you wrote no JSON.

## Step 4 &mdash; The agent in one call

`create_agent` runs the whole Module 4 loop: the model asks for a tool, the tool runs, the result goes
back into the messages, and the loop ends when the model answers with no tool call.

`recursion_limit` is your Day 1 step budget. It counts graph steps: **2 for each tool call, plus 2**.
So 12 allows up to five tool calls.

In [ ]:
from langchain.agents import create_agent

SYSTEM = ("You are AskOps, an assistant for on-call engineers. Answer only from tool results, "
          "in at most 6 lines. Cite runbook ids. If no tool helps, say so.")
agent = create_agent(llm, tools=tools, system_prompt=SYSTEM)

Q1 = "Payments is returning 502s since the 14:00 release. What do I do?"
trace(agent.invoke({"messages": [("user", Q1)]}, {"recursion_limit": 12}))

**You should see:** ACTION lines, such as `search_runbooks` and then `get_runbook` for RB-101, then
an answer that cites RB-101, then the steps and tokens. The model may choose a different order from
your Day 1 run. That is normal: a real model picks its path at run time.

## Step 5 &mdash; Stop an agent that needs too many steps

With a limit of 4, the agent may make only **one** tool call. If it needs more, LangGraph stops it
with `GraphRecursionError`. Catch it, and tell the user something useful.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    trace(agent.invoke({"messages": [("user", Q1)]}, {"recursion_limit": 4}))
except GraphRecursionError:
    print("Stopped: the step limit was reached. Ask a person, or raise the limit.")

**You should see:** usually the *Stopped* line, because a good answer needs more than one tool call.
If the model answered after one call, it stayed inside the limit.

## Step 6 &mdash; Stream the answer

`invoke` shows nothing until the end. `stream` with `stream_mode="messages"` gives you the answer a
few words at a time, while the model writes it. That is what a chat window needs.

In [ ]:
Q2 = "Logins are slow this morning. Is there a runbook?"
for token, meta in agent.stream({"messages": [("user", Q2)]}, {"recursion_limit": 12},
                                stream_mode="messages"):
    if meta["langgraph_node"] == "tools":
        print(f"\n[tool {token.name} returned]")
    elif token.content:
        print(token.content, end="", flush=True)

**You should see:** a line for each tool result, then the answer appearing piece by piece. It should cite RB-201.

## The result &mdash; compare with your Day 1 agent

Run both Lab 4.4 questions once more, and keep the output. Then, in a **terminal**, run your Day 1
agent on the same questions (the commands are in this folder's `README.md`) and fill in the
comparison table there.

In [ ]:
for q in (Q1, Q2):
    print("Q:", q)
    trace(agent.invoke({"messages": [("user", q)]}, {"recursion_limit": 12}))
    print("-" * 70)

**You should see:** two answers, one citing RB-101 and one citing RB-201, each with its steps and
tokens. The tokens should be close to your Day 1 numbers, because the same messages go to the model.

LangChain now owns the loop, the message list and the tool schemas. You still decide **which tools
exist**, **what their docstrings say** and **which tools may write**.